In [ ]:
import pandas as pd

schedule_df = pd.read_csv('data/emirates_flight_schedule.csv')
departures_df = pd.read_csv('data/emirates_dxb_departures_2026-02-27.csv')

print(schedule_df.shape, departures_df.shape)
schedule_df.head()

In [4]:
departures_df

,flight_number,departure_time,arrival_airport,arrival_time,aircraft_type,terminal,gate,status
0,EK 767,2026-02-26 23:25+04:00,JNB (Jo'anna),2026-02-27 00:01+04:00,Boeing 777-300ER,3.0,B14,Departed
1,EK 801,2026-02-27 00:15+04:00,JED (Jeddah),2026-02-27 00:34+04:00,Airbus A380-800,3.0,A13,Departed
2,EK 807,2026-02-27 01:15+04:00,MED (Medina),2026-02-27 01:44+04:00,Boeing 777-300ER,3.0,B28,Departed
3,EK 853,2026-02-27 01:25+04:00,KWI (Kuwait City),2026-02-27 01:48+04:00,Airbus A350-900,3.0,A14,Departed
4,EK 815,2026-02-27 01:25+04:00,RUH (Riyadh),2026-02-27 01:53+04:00,Boeing 777-300ER,3.0,B4,Departed
...,...,...,...,...,...,...,...,...
238,EK 606,2026-02-27 22:00+04:00,KHI (Karachi),2026-02-27 22:30+04:00,Boeing 777-200LR,3.0,A22,Departed
239,EK 905,2026-02-27 22:15+04:00,AMM (Amman),2026-02-27 22:26+04:00,Airbus A350-900,3.0,B12,Departed
240,EK 374,2026-02-27 22:35+04:00,BKK (Bangkok),2026-02-27 23:06+04:00,Airbus A380-800,3.0,A8,Departed
241,EK 538,2026-02-27 22:50+04:00,AMD (Ahmedabad),2026-02-27 23:20+04:00,Airbus A350-900,3.0,B14,Departed


In [ ]:
print("=== schedule_df: destination_airport value counts ===")
print(schedule_df['destination_airport'].value_counts().to_string())

print("\n=== departures_df: arrival_airport value counts ===")
print(departures_df['arrival_airport'].value_counts().to_string())

In [6]:

print("\n=== departures_df: arrival_airport value counts ===")
print(departures_df['arrival_airport'].value_counts().to_string())


=== departures_df: arrival_airport value counts ===
arrival_airport
LHR (London)                  6
JNB (Jo'anna)                 5
CAI (Cairo)                   5
BOM (Mumbai)                  5
BKK (Bangkok)                 5
MLE (Malé)                    4
LGW (London)                  4
SIN (Singapore)               4
DEL (New Delhi)               4
KWI (Kuwait City)             4
MAA (Chennai)                 3
BAH (Manama)                  3
MEL (Melbourne)               3
JED (Jeddah)                  3
MXP (Milan)                   3
RUH (Riyadh)                  3
AMD (Ahmedabad)               3
AMS (Amsterdam)               3
DUB (Dublin)                  3
FRA (Frankfurt-am-Main)       3
CDG (Paris)                   3
BCN (Barcelona)               3
DME (Moscow)                  3
HYD (Hyderabad)               3
BLR (Bangalore)               3
IST (Istanbul)                3
MNL (Manila)                  3
DAC (Dhaka)                   3
KUL (Kuala Lumpur)            3
KHI

In [ ]:
dubai_airports = ['Dubai International Airport', 'Dubai']
schedule_filtered = schedule_df[~schedule_df['destination_airport'].isin(dubai_airports)]

print("=== schedule_filtered: destination_airport value counts ===")
print(schedule_filtered['destination_airport'].value_counts().to_string())

print("\n=== departures_df: arrival_airport value counts ===")
print(departures_df['arrival_airport'].value_counts().to_string())

print(f"\nTotal flights in schedule_filtered: {len(schedule_filtered)}")
print(f"Unique flight numbers: {schedule_filtered['flight_number'].nunique()}")

# Impact of the Missile War on Emirates Airline Operations at DXB

## The Big Question
**What is the effect of the missile conflict on Emirates Airline departures from Dubai International Airport (DXB)?**

### Dataset
- **Source**: Emirates DXB departure board data (collected)
- **Period**: February 20 -- March 10, 2026 (19 days)
- **~6,000 flights** with: flight number, scheduled departure, destination, aircraft type, terminal, gate, and flight status

### Analysis Roadmap
We decompose the big question into six focused investigations:

| # | Question | Why It Matters |
|---|----------|----------------|
| 1 | **Daily Operations Overview** | Establish the baseline and the full disruption timeline |
| 2 | **The Shock: February 28** | Measure how sudden and severe the disruption was |
| 3 | **Route Impact** | Which destinations were suspended? Any geographic patterns? |
| 4 | **Time of Day** | Did operating hours shift during or after the crisis? |
| 5 | **Fleet Composition** | Did Emirates change aircraft assignments or downsize? |
| 6 | **Flight Duration** | Can we detect airspace rerouting from the data? |
| 7 | **Recovery** | How fast did operations resume? Did they return to normal? |

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import datetime

# ── Plotting defaults ──
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ── Load data ──
df = pd.read_csv('data/emirates_dxb_departures_2026-02-20_to_2026-03-10.csv')

# Parse datetime — keep original +04:00 timezone so .dt.date gives Dubai local dates
df['departure_dt'] = pd.to_datetime(df['departure_time'])
df['date'] = df['departure_dt'].dt.date
df['hour'] = df['departure_dt'].dt.hour

# ── Define analysis periods ──
def assign_period(d):
    if d < datetime.date(2026, 2, 28):
        return 'Pre-Crisis (Feb 20-27)'
    elif d < datetime.date(2026, 3, 5):
        return 'Crisis (Feb 28-Mar 4)'
    else:
        return 'Recovery (Mar 5-10)'

df['period'] = df['date'].apply(assign_period)
PERIODS = ['Pre-Crisis (Feb 20-27)', 'Crisis (Feb 28-Mar 4)', 'Recovery (Mar 5-10)']

# ── Region mapping (every destination in the dataset) ──
region_map = {
    # Middle East (Gulf, Levant, Iran, Iraq)
    'AMM': 'Middle East', 'BAH': 'Middle East', 'BEY': 'Middle East',
    'BGW': 'Middle East', 'BND': 'Middle East', 'BSR': 'Middle East',
    'DMM': 'Middle East', 'EBL': 'Middle East', 'IFN': 'Middle East',
    'IKA': 'Middle East', 'ISU': 'Middle East', 'JED': 'Middle East',
    'KWI': 'Middle East', 'LRR': 'Middle East', 'MCT': 'Middle East',
    'MED': 'Middle East', 'MHD': 'Middle East', 'NBJ': 'Middle East',
    'NJF': 'Middle East', 'RUH': 'Middle East', 'SLL': 'Middle East',
    'SYZ': 'Middle East', 'TLV': 'Middle East', 'ULH': 'Middle East',
    # South Asia
    'AMD': 'South Asia', 'BLR': 'South Asia', 'BOM': 'South Asia',
    'CCJ': 'South Asia', 'CCU': 'South Asia', 'CMB': 'South Asia',
    'COK': 'South Asia', 'DAC': 'South Asia', 'DEL': 'South Asia',
    'HYD': 'South Asia', 'ISB': 'South Asia', 'KBL': 'South Asia',
    'KHI': 'South Asia', 'KTM': 'South Asia', 'LHE': 'South Asia',
    'LKO': 'South Asia', 'LYP': 'South Asia', 'MAA': 'South Asia',
    'MLE': 'South Asia', 'MUX': 'South Asia', 'PEW': 'South Asia',
    'SKT': 'South Asia', 'TRV': 'South Asia', 'UET': 'South Asia',
    # Southeast Asia
    'BKK': 'Southeast Asia', 'CEB': 'Southeast Asia', 'CGK': 'Southeast Asia',
    'CRK': 'Southeast Asia', 'DAD': 'Southeast Asia', 'DPS': 'Southeast Asia',
    'HAN': 'Southeast Asia', 'HKT': 'Southeast Asia', 'KBV': 'Southeast Asia',
    'KUL': 'Southeast Asia', 'LGK': 'Southeast Asia', 'MNL': 'Southeast Asia',
    'SGN': 'Southeast Asia', 'SIN': 'Southeast Asia',
    # East Asia
    'CAN': 'East Asia', 'HGH': 'East Asia', 'HKG': 'East Asia',
    'HND': 'East Asia', 'ICN': 'East Asia', 'KIX': 'East Asia',
    'NRT': 'East Asia', 'PEK': 'East Asia', 'PVG': 'East Asia',
    'SZX': 'East Asia', 'TPE': 'East Asia',
    # Europe (incl. Turkey, Cyprus, Russia, Caucasus)
    'AER': 'Europe', 'AMS': 'Europe', 'ARN': 'Europe', 'ATH': 'Europe',
    'BCN': 'Europe', 'BEG': 'Europe', 'BGY': 'Europe', 'BHX': 'Europe',
    'BLQ': 'Europe', 'BRU': 'Europe', 'BSL': 'Europe', 'BUD': 'Europe',
    'CDG': 'Europe', 'CPH': 'Europe', 'CTA': 'Europe', 'DME': 'Europe',
    'DUB': 'Europe', 'DUS': 'Europe', 'EDI': 'Europe', 'ESB': 'Europe',
    'EVN': 'Europe', 'FCO': 'Europe', 'FRA': 'Europe', 'GLA': 'Europe',
    'GVA': 'Europe', 'GYD': 'Europe', 'HAM': 'Europe', 'IAS': 'Europe',
    'IST': 'Europe', 'KRK': 'Europe', 'KUF': 'Europe', 'KZN': 'Europe',
    'LCA': 'Europe', 'LED': 'Europe', 'LGW': 'Europe', 'LHR': 'Europe',
    'LIS': 'Europe', 'LJU': 'Europe', 'LYS': 'Europe', 'MAD': 'Europe',
    'MAN': 'Europe', 'MCX': 'Europe', 'MRV': 'Europe', 'MUC': 'Europe',
    'MXP': 'Europe', 'NAP': 'Europe', 'NCE': 'Europe', 'NCL': 'Europe',
    'OSL': 'Europe', 'OTP': 'Europe', 'OVB': 'Europe', 'POZ': 'Europe',
    'PRG': 'Europe', 'PSA': 'Europe', 'RIX': 'Europe', 'SAW': 'Europe',
    'SJJ': 'Europe', 'SOF': 'Europe', 'STN': 'Europe', 'SVX': 'Europe',
    'SZG': 'Europe', 'TBS': 'Europe', 'TIA': 'Europe', 'UFA': 'Europe',
    'VCE': 'Europe', 'VIE': 'Europe', 'VKO': 'Europe', 'VNO': 'Europe',
    'VOG': 'Europe', 'WAW': 'Europe', 'ZAG': 'Europe', 'ZRH': 'Europe',
    # Central Asia
    'ALA': 'Central Asia', 'ASB': 'Central Asia', 'CIT': 'Central Asia',
    'DYU': 'Central Asia', 'NQZ': 'Central Asia', 'RMO': 'Central Asia',
    'SKD': 'Central Asia', 'TAS': 'Central Asia',
    # Africa (Sub-Saharan)
    'ABJ': 'Africa', 'ACC': 'Africa', 'ADD': 'Africa', 'ASM': 'Africa',
    'CKY': 'Africa', 'CPT': 'Africa', 'DAR': 'Africa', 'DSS': 'Africa',
    'DUR': 'Africa', 'EBB': 'Africa', 'HGA': 'Africa', 'HRE': 'Africa',
    'JIB': 'Africa', 'JNB': 'Africa', 'LOS': 'Africa', 'LUN': 'Africa',
    'MBA': 'Africa', 'MRU': 'Africa', 'NBO': 'Africa', 'SEZ': 'Africa',
    'ZNZ': 'Africa',
    # North Africa
    'ALG': 'North Africa', 'CAI': 'North Africa', 'CMN': 'North Africa',
    'TUN': 'North Africa',
    # North America
    'BOS': 'North America', 'DFW': 'North America', 'EWR': 'North America',
    'IAD': 'North America', 'IAH': 'North America', 'JFK': 'North America',
    'LAX': 'North America', 'MCO': 'North America', 'MIA': 'North America',
    'ORD': 'North America', 'SEA': 'North America', 'SFO': 'North America',
    'YUL': 'North America', 'YYZ': 'North America',
    # South America
    'GIG': 'South America', 'GRU': 'South America', 'UIO': 'South America',
    # Oceania
    'ADL': 'Oceania', 'AKL': 'Oceania', 'BNE': 'Oceania',
    'MEL': 'Oceania', 'PER': 'Oceania', 'SYD': 'Oceania',
}
df['region'] = df['arrival_airport'].map(region_map).fillna('Other')

# ── Aircraft categories ──
def categorize_aircraft(ac):
    if pd.isna(ac): return 'Unknown'
    ac = str(ac)
    if 'A380' in ac: return 'Airbus A380'
    if '777'  in ac: return 'Boeing 777'
    if '787'  in ac: return 'Boeing 787'
    if 'A350' in ac: return 'Airbus A350'
    if '737'  in ac: return 'Boeing 737'
    return 'Other'

df['aircraft_cat'] = df['aircraft_type'].apply(categorize_aircraft)

# ── Convenience subsets ──
pre_df      = df[df['period'] == 'Pre-Crisis (Feb 20-27)']
crisis_df   = df[df['period'] == 'Crisis (Feb 28-Mar 4)']
recovery_df = df[df['period'] == 'Recovery (Mar 5-10)']

# ── Summary ──
print(f"Loaded {len(df):,} flights  |  {df['date'].min()} to {df['date'].max()}")
print(f"\nStatus breakdown:")
for status, count in df['status'].value_counts().items():
    pct = count / len(df) * 100
    print(f"  {status:12s}: {count:5,} ({pct:.1f}%)")
print(f"\nFlights by period:")
for p in PERIODS:
    n = (df['period'] == p).sum()
    print(f"  {p}: {n:,}")
unmapped = df[df['region'] == 'Other']['arrival_airport'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped airports: {', '.join(sorted(unmapped))}")

## 1. Daily Operations Overview

**Why this step:** Before diving into specifics, we need the big picture. A daily stacked bar chart of all flights by status immediately reveals:
- The normal operating baseline (~385 flights/day)
- *When* the crisis hit and how long it lasted
- Whether recovery reached baseline levels

**What to look for in the chart below:**
- **Green bars** (Departed) = normal operations
- **Red bars** (Canceled) = disruption
- The **shaded zone** marks the crisis window (Feb 28 -- Mar 4)
- Compare total bar height before, during, and after the crisis

In [ ]:
# ── Daily flight count by status ──
daily_status = df.groupby(['date', 'status']).size().unstack(fill_value=0)

# Order statuses logically: normal operations first, cancellations last
status_order = ['Departed', 'Delayed', 'Boarding', 'GateClosed', 'Expected', 'Canceled']
daily_status = daily_status.reindex(
    columns=[s for s in status_order if s in daily_status.columns], fill_value=0
)

status_colors = {
    'Departed': '#27ae60', 'Delayed': '#f39c12', 'Boarding': '#3498db',
    'GateClosed': '#9b59b6', 'Expected': '#1abc9c', 'Canceled': '#e74c3c',
}

fig, ax = plt.subplots(figsize=(16, 7))
daily_status.plot(
    kind='bar', stacked=True, ax=ax,
    color=[status_colors.get(s, '#95a5a6') for s in daily_status.columns],
    edgecolor='white', linewidth=0.5, width=0.85,
)

# Shade the crisis window
dates_list = list(daily_status.index)
try:
    ci = dates_list.index(datetime.date(2026, 2, 28))
    ce = dates_list.index(datetime.date(2026, 3, 4))
    ax.axvspan(ci - 0.5, ce + 0.5, alpha=0.08, color='red', zorder=0)
    ax.text(ci + (ce - ci) / 2, ax.get_ylim()[1] * 0.95, 'CRISIS WINDOW',
            ha='center', fontsize=11, color='red', fontweight='bold', alpha=0.5)
except ValueError:
    pass

ax.set_title('Emirates DXB Departures: Daily Flight Count by Status',
             fontsize=16, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Number of Flights')
ax.legend(title='Status', bbox_to_anchor=(1.02, 1), loc='upper left')

labels = [d.strftime('%b %d\n%a') for d in daily_status.index]
ax.set_xticklabels(labels, rotation=0, ha='center', fontsize=8)

plt.tight_layout()
plt.show()

# ── Summary table ──
print("\nDaily Cancellation Summary:")
print(f"{'Date':<12} {'Total':>6} {'Canceled':>9} {'Rate':>7}")
print("-" * 38)
for d in sorted(daily_status.index):
    total = int(daily_status.loc[d].sum())
    canc  = int(daily_status.loc[d].get('Canceled', 0))
    rate  = canc / total * 100 if total > 0 else 0
    flag  = " <<<" if rate > 50 else ""
    print(f"{str(d):<12} {total:>6} {canc:>9} {rate:>6.1f}%{flag}")

## 2. The Shock: What Happened on February 28?

**Why this step:** The daily chart shows Feb 28 as the inflection point -- cancellations jumped from near-zero to ~44% in a single day. But *how* did it unfold within those 24 hours?

- If cancellations ramped up **gradually** hour by hour --> Emirates had advance warning and wound down operations (**planned suspension**)
- If there was a **sharp cutoff** at a specific hour --> the disruption was externally imposed (**sudden shock** -- airspace closure, government order)

We also want the **last flight that departed** before the full shutdown, and the **first flight back** -- these are powerful narrative data points that anchor the timeline.

In [ ]:
# ── Hour-by-hour analysis of February 28 ──
feb28 = df[df['date'] == datetime.date(2026, 2, 28)]

hourly_total    = feb28.groupby('hour').size().reindex(range(24), fill_value=0)
hourly_departed = feb28[feb28['status'] == 'Departed'].groupby('hour').size().reindex(range(24), fill_value=0)
hourly_canceled = feb28[feb28['status'] == 'Canceled'].groupby('hour').size().reindex(range(24), fill_value=0)
hourly_other    = hourly_total - hourly_departed - hourly_canceled
cancel_rate     = np.where(hourly_total > 0, hourly_canceled / hourly_total * 100, 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: stacked bar by hour
ax1.bar(range(24), hourly_departed, color='#27ae60', label='Departed')
ax1.bar(range(24), hourly_canceled, bottom=hourly_departed, color='#e74c3c', label='Canceled')
ax1.bar(range(24), hourly_other, bottom=hourly_departed + hourly_canceled,
        color='#95a5a6', label='Other')
ax1.set_xlabel('Hour of Day (Dubai Time)')
ax1.set_ylabel('Number of Flights')
ax1.set_title('Feb 28: Flights by Hour', fontsize=14, fontweight='bold')
ax1.set_xticks(range(0, 24))
ax1.legend()

# Right: cancellation rate by hour
ax2.plot(range(24), cancel_rate, 'o-', color='#e74c3c', linewidth=2, markersize=7)
ax2.fill_between(range(24), cancel_rate, alpha=0.15, color='#e74c3c')
ax2.set_xlabel('Hour of Day (Dubai Time)')
ax2.set_ylabel('Cancellation Rate (%)')
ax2.set_title('Feb 28: Cancellation Rate by Hour', fontsize=14, fontweight='bold')
ax2.set_ylim(-5, 105)
ax2.set_xticks(range(0, 24))
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.4, linewidth=1)

plt.tight_layout()
plt.show()

# ── Day-by-day transition ──
print("\n=== Day-by-Day Transition into Shutdown ===")
for d in [datetime.date(2026, 2, 27), datetime.date(2026, 2, 28),
          datetime.date(2026, 3, 1), datetime.date(2026, 3, 2)]:
    day_df = df[df['date'] == d]
    if len(day_df) == 0:
        continue
    dep  = (day_df['status'] == 'Departed').sum()
    canc = (day_df['status'] == 'Canceled').sum()
    print(f"  {d}  |  {len(day_df):>3} flights  |  "
          f"{dep:>3} departed  |  {canc:>3} canceled ({canc/len(day_df)*100:.1f}%)")

# ── Last flight out & first flight back ──
# Last departed flight on Feb 28
last_feb28_dep = df[(df['date'] == datetime.date(2026, 2, 28)) &
                    (df['status'] == 'Departed')].sort_values('departure_dt')
if len(last_feb28_dep) > 0:
    last = last_feb28_dep.iloc[-1]
    print(f"\n=== Last Flight Out on Feb 28 ===")
    print(f"  Flight:      {last['flight_number']}")
    print(f"  Destination: {last['arrival_airport']} ({region_map.get(last['arrival_airport'], '?')})")
    print(f"  Departed:    {last['departure_time']}")
    print(f"  Aircraft:    {last['aircraft_type']}")

# First departed flight after Feb 28
post = df[(df['date'] > datetime.date(2026, 2, 28)) &
          (df['status'] == 'Departed')].sort_values('departure_dt')
if len(post) > 0 and len(last_feb28_dep) > 0:
    first = post.iloc[0]
    print(f"\n=== First Flight Back After Shutdown ===")
    print(f"  Flight:      {first['flight_number']}")
    print(f"  Destination: {first['arrival_airport']} ({region_map.get(first['arrival_airport'], '?')})")
    print(f"  Departed:    {first['departure_time']}")
    print(f"  Aircraft:    {first['aircraft_type']}")

    gap_hours = (first['departure_dt'] - last['departure_dt']).total_seconds() / 3600
    print(f"\n  Shutdown duration: {gap_hours:.1f} hours ({gap_hours/24:.1f} days)")

## 3. Route Impact: Which Destinations Were Hit?

**Why this step:** Not all routes are affected equally in a conflict. The pattern reveals the *nature* of the disruption:

- If **only Middle Eastern routes** were cancelled --> regional **airspace closures**
- If **everything stopped** --> the threat was to **DXB itself** (airport closure)
- If routes to **certain countries** survived --> political or diplomatic factors at play

We examine:
1. Which routes had **100% cancellation** during the crisis?
2. Which routes **kept flying**?
3. Does the pattern cluster **geographically**?

In [ ]:
# ── Route-level cancellation during crisis ──
route_crisis = crisis_df.groupby('arrival_airport').agg(
    total=('flight_number', 'count'),
    canceled=('status', lambda x: (x == 'Canceled').sum()),
)
route_crisis['cancel_rate'] = (route_crisis['canceled'] / route_crisis['total'] * 100).round(1)
route_crisis['region'] = route_crisis.index.map(lambda x: region_map.get(x, 'Other'))

# 100% cancelled routes
fully_canceled = route_crisis[route_crisis['cancel_rate'] == 100].sort_values('total', ascending=False)
print(f"=== Routes with 100% Cancellation During Crisis ===")
print(f"    {len(fully_canceled)} out of {len(route_crisis)} routes were FULLY suspended")

# Routes that maintained at least some service
kept_flying = route_crisis[route_crisis['cancel_rate'] < 100].sort_values('cancel_rate')
print(f"\n=== Routes That Maintained Some Service ===")
print(f"    {len(kept_flying)} routes kept at least partial operations\n")
if len(kept_flying) > 0:
    print(f"{'Airport':<8} {'Region':<18} {'Total':>6} {'Canceled':>9} {'Rate':>7}")
    print("-" * 52)
    for airport, row in kept_flying.iterrows():
        print(f"{airport:<8} {row['region']:<18} {int(row['total']):>6} "
              f"{int(row['canceled']):>9} {row['cancel_rate']:>6.1f}%")

# ── Regional cancellation rates ──
region_crisis = crisis_df.groupby('region').agg(
    total=('flight_number', 'count'),
    canceled=('status', lambda x: (x == 'Canceled').sum()),
)
region_crisis['cancel_rate'] = (region_crisis['canceled'] / region_crisis['total'] * 100).round(1)
region_crisis = region_crisis.sort_values('cancel_rate', ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: cancellation rate by region
colors_bar = ['#e74c3c' if r > 80 else '#f39c12' if r > 50 else '#27ae60'
              for r in region_crisis['cancel_rate']]
bars = ax1.barh(region_crisis.index, region_crisis['cancel_rate'],
                color=colors_bar, edgecolor='white')
ax1.set_xlabel('Cancellation Rate (%)')
ax1.set_title('Crisis Period: Cancellation Rate by Region', fontsize=14, fontweight='bold')
ax1.set_xlim(0, 109)
for bar, rate in zip(bars, region_crisis['cancel_rate']):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
             f'{rate:.0f}%', va='center', fontsize=10)

# Right: surviving routes by region
if len(kept_flying) > 0:
    survived_regions = kept_flying.groupby('region').size().sort_values(ascending=True)
    ax2.barh(survived_regions.index, survived_regions.values, color='#27ae60', edgecolor='white')
    ax2.set_xlabel('Number of Routes That Kept Flying')
    ax2.set_title('Surviving Routes by Region', fontsize=14, fontweight='bold')
    for i, (region, count) in enumerate(survived_regions.items()):
        ax2.text(count + 0.2, i, str(count), va='center', fontsize=11)
else:
    ax2.text(0.5, 0.5, 'No routes maintained service', transform=ax2.transAxes,
             ha='center', va='center', fontsize=14, color='gray')
    ax2.set_title('Surviving Routes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n=== Key Observation ===")
total_routes = len(route_crisis)
pct_fully = len(fully_canceled) / total_routes * 100
print(f"  {pct_fully:.0f}% of routes ({len(fully_canceled)}/{total_routes}) were fully cancelled.")
if pct_fully > 90:
    print("  This suggests the disruption was AIRPORT-WIDE (DXB closure),")
    print("  not route-specific (airspace restrictions on certain corridors).")

## 4. Time of Day: When Is Emirates Flying?

**Why this step:** Emirates operates a hub-and-spoke model with specific "wave" departures timed for passenger connections. If the crisis changed *when* flights depart -- not just *how many* -- it reveals deeper operational restructuring (e.g., avoiding certain hours, concentrating departures into safe windows).

**What to look for:**
- **Pre-crisis**: the normal hourly pattern (peaks at connection bank times)
- **Crisis**: are the few surviving departures clustered at certain hours?
- **Recovery**: has the hourly pattern returned to normal, or is it shifted?

In [ ]:
# ── Time of day comparison across periods ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, period in zip(axes, PERIODS):
    pdata = df[df['period'] == period]

    dep_hourly  = pdata[pdata['status'] == 'Departed'].groupby('hour').size()
    canc_hourly = pdata[pdata['status'] == 'Canceled'].groupby('hour').size()

    hours    = range(24)
    dep_vals  = [dep_hourly.get(h, 0)  for h in hours]
    canc_vals = [canc_hourly.get(h, 0) for h in hours]

    ax.bar(hours, dep_vals, color='#27ae60', label='Departed', alpha=0.85)
    ax.bar(hours, canc_vals, bottom=dep_vals, color='#e74c3c', label='Canceled', alpha=0.85)

    short_title = period.split('(')[0].strip()
    ax.set_title(f'{short_title}\n({len(pdata):,} flights)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Hour of Day')
    ax.set_xticks(range(0, 24, 3))
    if ax is axes[0]:
        ax.set_ylabel('Flights (total over period)')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle('Flight Departures by Hour of Day', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# ── Peak hours comparison ──
print("\n=== Peak Departure Hours (Departed flights only) ===")
for period in ['Pre-Crisis (Feb 20-27)', 'Recovery (Mar 5-10)']:
    pdata = df[(df['period'] == period) & (df['status'] == 'Departed')]
    hourly = pdata.groupby('hour').size()
    top3 = hourly.nlargest(3)
    short = period.split('(')[0].strip()
    print(f"\n  {short}:")
    for h, count in top3.items():
        print(f"    {h:02d}:00 -- {count} departures")

## 5. Fleet Composition: Did the Aircraft Change?

**Why this step:** Aircraft type is a proxy for capacity and revenue impact:

| Aircraft | Typical Seats | Role |
|----------|-------------|------|
| Airbus A380 | ~500 | Flagship, highest capacity |
| Boeing 777 | ~350 | Long-haul workhorse |
| Airbus A350 / Boeing 787 | ~250-300 | Efficient long-haul |
| Boeing 737 | ~160 | Short/medium-haul (often flydubai codeshares) |

**Key questions:**
- Were **wide-bodies** (A380, 777) disproportionately cancelled? If so, the revenue impact is larger than flight counts suggest.
- Did Emirates **downsize aircraft** on the same routes in recovery (e.g., swapping A380 for 777)?
- Did the **fleet mix** (share of each type) shift between pre-crisis and recovery?

In [ ]:
# ── Fleet composition by period ──
fleet_by_period = df.groupby(['period', 'aircraft_cat']).size().unstack(fill_value=0)
fleet_pct = fleet_by_period.div(fleet_by_period.sum(axis=1), axis=0) * 100

# ── Crisis cancellation by aircraft type ──
crisis_fleet = crisis_df.groupby('aircraft_cat').agg(
    total=('flight_number', 'count'),
    canceled=('status', lambda x: (x == 'Canceled').sum()),
    departed=('status', lambda x: (x == 'Departed').sum()),
)
crisis_fleet['cancel_rate'] = (crisis_fleet['canceled'] / crisis_fleet['total'] * 100).round(1)

aircraft_colors = {
    'Airbus A380': '#e74c3c', 'Boeing 777': '#3498db', 'Boeing 787': '#2ecc71',
    'Airbus A350': '#f39c12', 'Boeing 737': '#9b59b6', 'Other': '#95a5a6',
    'Unknown': '#bdc3c7',
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: fleet mix comparison
fleet_pct_plot = fleet_pct.reindex(PERIODS).fillna(0)
fleet_pct_plot.plot(
    kind='bar', ax=ax1, edgecolor='white',
    color=[aircraft_colors.get(c, '#95a5a6') for c in fleet_pct_plot.columns],
)
ax1.set_title('Fleet Mix by Period (% of flights)', fontsize=14, fontweight='bold')
ax1.set_ylabel('% of Flights')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=15)
ax1.legend(title='Aircraft', fontsize=8, bbox_to_anchor=(1.02, 1))

# Right: cancellation rate by aircraft during crisis
crisis_fleet_sorted = crisis_fleet.sort_values('cancel_rate', ascending=True)
bars = ax2.barh(
    crisis_fleet_sorted.index, crisis_fleet_sorted['cancel_rate'],
    color=[aircraft_colors.get(a, '#95a5a6') for a in crisis_fleet_sorted.index],
    edgecolor='white',
)
ax2.set_xlabel('Cancellation Rate (%)')
ax2.set_title('Crisis: Cancellation Rate by Aircraft Type', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 109)
for bar, rate in zip(bars, crisis_fleet_sorted['cancel_rate']):
    ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
             f'{rate:.0f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

# ── Aircraft swaps on key routes ──
print("\n=== Aircraft Assignment: Pre-Crisis vs Recovery (Top 20 Routes) ===")
print(f"{'Route':<6} {'Pre-Crisis':<25} {'Recovery':<25} {'Change?'}")
print("-" * 75)
top_routes = pre_df['arrival_airport'].value_counts().head(20).index
for route in top_routes:
    pre_ac = pre_df[pre_df['arrival_airport'] == route]['aircraft_cat'].mode()
    rec_ac = recovery_df[recovery_df['arrival_airport'] == route]['aircraft_cat'].mode()
    pre_str = pre_ac.iloc[0] if len(pre_ac) > 0 else 'N/A'
    rec_str = rec_ac.iloc[0] if len(rec_ac) > 0 else 'Not resumed'
    changed = '<-- CHANGED' if (pre_str != rec_str and rec_str != 'Not resumed'
                                and pre_str != 'N/A') else ''
    print(f"{route:<6} {pre_str:<25} {rec_str:<25} {changed}")

## 6. Flight Duration: A Data Quality Investigation

**Why this step:** A natural follow-up question is whether **flight durations increased** during the crisis, which would suggest aircraft were being rerouted around closed airspace (longer flight paths).

Before analyzing, we must verify that the `arrival_time` column actually represents **arrival at the destination**. Flight tracking data often has surprising column semantics -- let's check.

In [ ]:
# ── Investigate the arrival_time column ──
df['arrival_dt'] = pd.to_datetime(df['arrival_time'])
df['apparent_duration_min'] = (df['arrival_dt'] - df['departure_dt']).dt.total_seconds() / 60

departed_only = df[df['status'] == 'Departed']

print("=== 'Duration' Statistics (departure_time -> arrival_time) ===")
print(departed_only['apparent_duration_min'].describe().round(1).to_string())

# Spot-check known long-haul routes
print("\n=== Spot Check: Known Long-Haul Routes ===")
long_haul_expected = {
    'SYD': '~14h', 'JFK': '~14h', 'LAX': '~16h',
    'LHR': '~7h',  'JNB': '~8h',  'BKK': '~6h',
}
for airport, expected in long_haul_expected.items():
    rd = departed_only[departed_only['arrival_airport'] == airport]
    if len(rd) > 0:
        avg = rd['apparent_duration_min'].mean()
        print(f"  {airport} (expected {expected}):  data shows {avg:.0f} min ({avg/60:.1f}h)")

print("\n" + "=" * 65)
print("  FINDING: 'arrival_time' does NOT represent arrival at")
print("  the destination. The recorded durations are far too short")
print("  for actual flight times (e.g., ~30 min for a 14-hour flight).")
print("")
print("  Most likely, 'arrival_time' represents the actual gate")
print("  departure / pushback time, while 'departure_time' is the")
print("  scheduled departure. The gap = departure delay.")
print("")
print("  => Flight duration / rerouting analysis is NOT possible")
print("     with this dataset.")
print("  => We CAN interpret the gap as departure delay.")
print("=" * 65)

# ── Use the gap as departure delay ──
print("\n=== Average Departure Delay by Period (minutes) ===")
delay_stats = departed_only.groupby('period')['apparent_duration_min'].agg(
    ['count', 'mean', 'median', 'max']
).reindex(PERIODS).round(1)
delay_stats.columns = ['Flights', 'Mean Delay', 'Median Delay', 'Max Delay']
print(delay_stats.to_string())

## 7. Recovery Analysis: Did Operations Return to Normal?

**Why this step:** The crisis window (Feb 28 -- Mar 4) saw a near-total shutdown. Recovery speed tells a critical story:

- **Fast recovery** --> temporary suspension, infrastructure intact, demand ready
- **Slow / partial recovery** --> lasting damage, route abandonment, reduced demand

A key observation from the daily chart: cancellation rate dropped sharply from 73% (Mar 4) to 6.7% (Mar 5). But daily flight **volume** also dropped from ~385 pre-crisis to ~190 in recovery. That means: **flights are running again, but at roughly half capacity.**

**Sub-questions:**
- How does recovery volume compare to the pre-crisis baseline?
- Which routes came back first -- short-haul regional or long-haul?
- Are there routes that operated pre-crisis but **never resumed** in this dataset?
- Is the lower volume due to fewer routes, fewer frequencies, or both?

In [ ]:
# ── Volume comparison ──
pre_daily_avg = pre_df.groupby('date').size().mean()
rec_daily_avg = recovery_df.groupby('date').size().mean()

print(f"=== Volume Comparison ===")
print(f"  Pre-crisis daily avg:  {pre_daily_avg:.0f} flights/day")
print(f"  Recovery daily avg:    {rec_daily_avg:.0f} flights/day")
print(f"  Recovery vs pre-crisis: {rec_daily_avg / pre_daily_avg * 100:.1f}%")

# Route network
pre_routes     = set(pre_df['arrival_airport'].unique())
rec_routes_dep = set(recovery_df[recovery_df['status'] != 'Canceled']['arrival_airport'].unique())

print(f"\n=== Route Network ===")
print(f"  Pre-crisis unique routes:  {len(pre_routes)}")
print(f"  Recovery operating routes: {len(rec_routes_dep)}")
print(f"  Routes NOT resumed:        {len(pre_routes - rec_routes_dep)}")

# ── Timeline: area chart with recovery context ──
fig, ax = plt.subplots(figsize=(16, 7))

dates = sorted(df['date'].unique())
daily_dep   = df[df['status'] == 'Departed'].groupby('date').size()
daily_total = df.groupby('date').size()

dep_vals   = [daily_dep.get(d, 0)   for d in dates]
total_vals = [daily_total.get(d, 0) for d in dates]

x = list(range(len(dates)))
ax.fill_between(x, 0, dep_vals, color='#27ae60', alpha=0.7, label='Departed')
ax.fill_between(x, dep_vals, total_vals, color='#e74c3c', alpha=0.7, label='Canceled / Other')

# Pre-crisis average line
ax.axhline(y=pre_daily_avg, color='#2c3e50', linestyle='--', linewidth=2,
           label=f'Pre-crisis avg ({pre_daily_avg:.0f}/day)')

# Shade crisis window
crisis_idx = [i for i, d in enumerate(dates)
              if datetime.date(2026, 2, 28) <= d <= datetime.date(2026, 3, 4)]
if crisis_idx:
    ax.axvspan(crisis_idx[0] - 0.5, crisis_idx[-1] + 0.5, alpha=0.08, color='red', zorder=0)

ax.set_xticks(x)
ax.set_xticklabels([d.strftime('%b %d') for d in dates], rotation=45, ha='right')
ax.set_ylabel('Number of Flights')
ax.set_title('Emirates DXB: Full Operations Timeline & Recovery',
             fontsize=16, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

# ── Which routes came back first? ──
first_back = (recovery_df[recovery_df['status'] != 'Canceled']
              .groupby('arrival_airport')
              .agg(first_flight=('departure_dt', 'min'),
                   num_flights=('flight_number', 'count'))
              .sort_values('first_flight'))
first_back['region'] = first_back.index.map(lambda x: region_map.get(x, 'Other'))
first_back['day'] = first_back['first_flight'].dt.date

print(f"\n=== Route Resumption Order (first 25 routes back) ===")
print(f"{'Date':<12} {'Airport':<7} {'Region':<18} {'Flights':>8}")
print("-" * 48)
for airport, row in first_back.head(25).iterrows():
    print(f"{row['day']}   {airport:<7} {row['region']:<18} {row['num_flights']:>6}")

# ── Routes that never recovered ──
never_recovered = sorted(pre_routes - rec_routes_dep)
if never_recovered:
    print(f"\n=== Routes That Never Resumed in Dataset ({len(never_recovered)} routes) ===")
    by_region = {}
    for r in never_recovered:
        rgn = region_map.get(r, 'Other')
        by_region.setdefault(rgn, []).append(r)
    for rgn in sorted(by_region):
        airports = ', '.join(sorted(by_region[rgn]))
        print(f"  {rgn}: {airports}")
    print(f"\n  Note: with only {len(rec_routes_dep)} days of recovery data, some weekly")
    print(f"  routes may appear 'missing' simply because their day hasn't come yet.")

# ── Recovery by region ──
region_first = first_back.groupby('region')['day'].min().sort_values()
print(f"\n=== First Recovery Date by Region ===")
for rgn, date in region_first.items():
    print(f"  {date}  |  {rgn}")

## Key Findings Summary

### The Shock
- The crisis struck **suddenly on Feb 28**, escalating from near-zero cancellations to a **100% shutdown on Mar 1** within roughly 36 hours
- The disruption lasted **5 full days** (Feb 28 -- Mar 4)
- **~1,550 flights** were cancelled out of ~6,000 in the dataset

### The Impact
- The shutdown was **airport-wide**: virtually all routes and aircraft types were affected equally
- This indicates the disruption was caused by a **threat to DXB itself** (airport/airspace closure), not targeted route suspensions
- Wide-body aircraft (A380, 777) -- representing Emirates' highest-capacity routes -- were cancelled at the same rate, amplifying the revenue impact

### The Recovery
- Operations resumed sharply on **Mar 5** (cancellation rate dropped from 73% to ~7% overnight)
- However, daily flight volume settled at roughly **half the pre-crisis level** (~190 vs ~385 flights/day)
- Multiple pre-crisis routes had **not yet resumed** by Mar 10
- The reduced volume suggests ongoing capacity management, continued airspace restrictions, or reduced passenger demand

### Data Limitations
- `arrival_time` does not represent actual arrival at destination -- flight duration and rerouting analysis is not possible with this dataset
- The dataset covers only 19 days -- longer-term recovery trends cannot be assessed
- With only 6 days of recovery data, some infrequent (weekly) routes may appear as "never recovered" prematurely